## Hovering-Intruder Avoidance

Two UAVs, both running ArduPilot's ADS-B avoidance (`AVD_*` in `simulator/params/vehicle.parm`):
- **Transit** (sysid=1, BLUE): flies 100 m north at 6 m altitude and 10 m/s, then lands.
- **Intruder** (sysid=2, RED): takes off to 5 m halfway along that path and hovers there.

The red sphere marks the 5 m failsafe radius (`AVD_F_DIST_XY`) that the transit UAV must deviate around.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()


## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

base_homes = ENUPose.list([(5, 0, 0, 0), (5, 50, 0, 0)])
base_paths = [
    ENU.list([(0, 0, 0), (0, 0, 6), (0, 100, 6)]),
    ENU.list([(0, 0, 0), (0, 0, 5)]),
]

market_home = base_homes[1]
base_intruder_center = ENU(x=0, y=0, z=5)  # 15,15
market_enu_home = enu_origin.to_abs(market_home)
enu_intruder_center = market_enu_home.to_abs(base_intruder_center).unpose()
intruder_radius = 5  # 5


## Oracle

In [ ]:
orac = Oracle()

## Create Vehicles

In [ ]:
sysids = [1, 2]
models = [Model.IRIS, Model.IRIS]
colors = [Color.BLUE, Color.RED]

speeds = [10.0, 2.0]  # m/s
lands = [True, False]

for sysid, base_home, base_path, color, speed, land, model in zip(
    sysids, base_homes, base_paths, colors, speeds, lands, models, strict=True
):
    auto_plan = AutoPlan.from_relative_path(
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        relative_path=base_path,
        land=land,
        navigation_speed=speed,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
        model=model,
    )
    orac.add_vehicle(veh)


## Visualizer

### Gazebo

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)
gaz.markers.append(origin_gaz)

intruder_sphere_gaz = GazMarker(
    name="intruder_sphere",
    group="intruder_sphere",
    pos=enu_intruder_center,
    radius=intruder_radius,
    color=Color.RED,
    alpha=0.9,
)
gaz.markers.append(intruder_sphere_gaz)


### QGroundControl

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(name="origin", pos=gra_origin.unpose(), color=Color.WHITE)
qgc.markers.append(origin_qgc)


### No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)


## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    terminals=[SimProcess.LOGIC],
    verbose=1,
)

simulator.preview()


## Run

In [ ]:
simulator.run(timeout=120)


In [ ]:
orac.plot_trajectories();